# Round 2 — EDA and Feature Engineering\nCustomer Churn Insight. Data is synthetic practice data, not actual company data. Upload the five CSVs into the notebook working directory.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the five CSV files in the same folder
customers = pd.read_csv("customers.csv")
usage = pd.read_csv("usage.csv")
billing = pd.read_csv("billing.csv")
support = pd.read_csv("support.csv")
service = pd.read_csv("service.csv")

# Inspect shape, types, missing values, and summary statistics
for name, df in {"customers":customers, "usage":usage, "billing":billing, "support":support, "service":service}.items():
    print("\\n---", name, "---")
    print(df.shape)
    print(df.head())
    df.info()
    print(df.isna().sum())

# Standardize plan names and remove duplicate customer IDs
customers["Plan"] = customers["Plan"].astype(str).str.strip().str.title()
customers = customers.drop_duplicates(subset="CustomerID", keep="first")
usage = usage.drop_duplicates(subset="CustomerID", keep="first")
usage["MonthlyUsageGB"] = usage["MonthlyUsageGB"].fillna(usage["MonthlyUsageGB"].median())
billing = billing.drop_duplicates(subset="CustomerID", keep="first")
q1, q3 = billing["MonthlyCharges"].quantile([.25, .75])
iqr = q3 - q1
lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
billing["BillOutlierFlag"] = ((billing["MonthlyCharges"] < lower) | (billing["MonthlyCharges"] > upper)).astype(int)
billing["MonthlyCharges_Original"] = billing["MonthlyCharges"]
billing["MonthlyCharges"] = billing["MonthlyCharges"].clip(lower, upper)
support = support.drop_duplicates(subset="CustomerID", keep="first")
support["SupportHistoryMissing"] = support[["Tickets","ResolutionHours"]].isna().any(axis=1).astype(int)
support["Tickets"] = support["Tickets"].fillna(0)
support["ResolutionHours"] = support["ResolutionHours"].fillna(support["ResolutionHours"].median())
service = service.drop_duplicates(subset="CustomerID", keep="first")

# Merge customer-level tables
data = customers.merge(usage, on="CustomerID", how="left", suffixes=("","_usage"))
data = data.merge(billing, on="CustomerID", how="left", suffixes=("","_billing"))
data = data.merge(support, on="CustomerID", how="left", suffixes=("","_support"))
data = data.merge(service, on="CustomerID", how="left", suffixes=("","_service"))
for col in ["MonthlyCharges","Complaints","NetworkIssues"]:
    alt = col + "_billing" if col == "MonthlyCharges" else col + ("_support" if col=="Complaints" else "_service")
    if alt in data.columns:
        data[col] = data[alt].fillna(data[col])

data["ChurnFlag"] = (data["Churn"] == "Yes").astype(int)
data["TenureGroup"] = pd.cut(data["TenureMonths"], [0,12,24,48,100], labels=["0-12 months","13-24 months","25-48 months","49+ months"])
data["LowUsageFlag"] = (data["MonthlyUsageGB"] < 100).astype(int)
data["HighComplaintFlag"] = (data["Complaints"] >= 2).astype(int)
data["PaymentRiskFlag"] = (data["LatePaymentsLast4"] >= 2).astype(int)
data["EngagementRiskScore"] = data["LowUsageFlag"] + data["HighComplaintFlag"] + data["PaymentRiskFlag"] + (data["NetworkIssues"] >= 2).astype(int)

# EDA tables
print(data.describe(include="all").T)
for col in ["Plan","TenureGroup","Region","PaymentStatus"]:
    print("\\nChurn by", col)
    print(data.groupby(col, observed=False).agg(Customers=("CustomerID","count"), Churned=("ChurnFlag","sum"), ChurnRate=("ChurnFlag","mean")))
data["ComplaintGroup"] = np.where(data["Complaints"] >= 2, "2+ complaints", "0-1 complaints")
data["UsageGroup"] = pd.cut(data["MonthlyUsageGB"], [-1,100,200,350,10000], labels=["<100 GB","100-200 GB","201-350 GB",">350 GB"])

# Visualizations
sns.set_theme()
for col, title in [("Plan","Churn by Plan"),("TenureGroup","Churn by Tenure"),("Region","Churn by Region"),("PaymentStatus","Churn by Payment Status"),("ComplaintGroup","Churn by Complaints"),("UsageGroup","Churn by Usage")]:
    plt.figure(figsize=(8,4))
    sns.barplot(data=data, x=col, y="ChurnFlag", errorbar=None)
    plt.title(title); plt.ylabel("Churn rate"); plt.xticks(rotation=20); plt.tight_layout(); plt.show()

# Feature screening (correlation is not causation)
numeric = data.select_dtypes(include="number")
print(numeric.corr()["ChurnFlag"].drop("ChurnFlag").sort_values(key=lambda x: x.abs(), ascending=False))

data.to_csv("cleaned_customer_churn.csv", index=False)
data.to_excel("cleaned_customer_churn.xlsx", index=False)
